Can a sparse variational GP provide a practical, uncertainty-aware, continuously updating digital twin of a coastal beach environment using heterogeneous multi-source elevation data?

In [ ]:
# ============================================================
# SECTION 0 - LOCAL DEPENDENCY CHECK
# ============================================================
# Run this notebook from the project folder so relative data paths resolve correctly.
# Missing packages are reported only; install them in the notebook environment
# before continuing.

import importlib.util

_required_packages = ["numpy", "pandas", "tensorflow", "gpflow", "matplotlib", "scipy", "sklearn"]
_missing_packages = [pkg for pkg in _required_packages if importlib.util.find_spec(pkg) is None]

if _missing_packages:
    print("Missing packages:", ", ".join(_missing_packages))
    print("Install them in your environment, then restart the kernel and rerun the notebook.")
else:
    print("Core notebook dependencies are available.")


In [ ]:
# ============================================================
# SECTION 1 - IMPORTS
# ============================================================

import os
import glob
import json
import re
import sys
from datetime import datetime as _datetime, timezone as _timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import gpflow
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm as _norm
from scipy.spatial import KDTree
from sklearn.preprocessing import StandardScaler

print("TensorFlow :", tf.__version__)
print("GPflow     :", gpflow.__version__)
print("NumPy      :", np.__version__)


In [ ]:
# ===========================================================
# SECTION 2 - SHARED PROJECT-RELATIVE CONFIG
# ===========================================================

from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Codes").is_dir() and (candidate / "Datasets").is_dir() and (candidate / "Models").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing Codes, Datasets, and Models")


PROJECT_ROOT = find_project_root()
DATASETS = PROJECT_ROOT / "Datasets"
MODELS = PROJECT_ROOT / "Models"

# Manual choice: set BEACH to "bass" or "bicentennial" before running.
BEACH = "bicentennial"

CONFIG = {
    "bass": {
        "LABEL": "Bass",
        "NOAA_CSV": DATASETS / "Extracted" / "Bass CSV" / "NOAA_all_surveys_combined.csv",
        "RTK_DIR": DATASETS / "Sunday Data" / "Bass",
        "OUT_DIR": MODELS / "BASS 3",
        "LAST_NOAA_DATE_STR": "2024-11-14",
    },
    "bicentennial": {
        "LABEL": "Bicentennial",
        "NOAA_CSV": DATASETS / "Extracted" / "Bicent CSV" / "NOAA_all_surveys_combined.csv",
        "RTK_DIR": DATASETS / "Sunday Data" / "BicantP",
        "OUT_DIR": MODELS / "BICENTP 2",
        "LAST_NOAA_DATE_STR": "2024-11-14",
    },
}

if BEACH not in CONFIG:
    raise ValueError(f"Unknown BEACH {BEACH!r}. Choose one of: {', '.join(CONFIG)}")

BEACH_LABEL = CONFIG[BEACH]["LABEL"]
NOAA_CSV = CONFIG[BEACH]["NOAA_CSV"]
RTK_DIR = CONFIG[BEACH]["RTK_DIR"]
OUT_DIR = CONFIG[BEACH]["OUT_DIR"]
LAST_NOAA_DATE_STR = CONFIG[BEACH]["LAST_NOAA_DATE_STR"]

OUT_DIR.mkdir(parents=True, exist_ok=True)

for label, path in [("NOAA_CSV", NOAA_CSV), ("RTK_DIR", RTK_DIR)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")

print("Project root:", PROJECT_ROOT)
print("Beach:", BEACH_LABEL)
print("NOAA CSV:", NOAA_CSV)
print("RTK dir:", RTK_DIR)
print("Output dir:", OUT_DIR)

# Hyper-parameters
TARGET_H = 100
TARGET_W = 150
M_INDUCING = 500
BATCH_SIZE = 4096
TRAIN_STEPS = 3000
RTK_UPDATE_STEPS = 1000

# Reproducibility controls. These make the inducing-point sample and
# TensorFlow initialisation repeatable for future runs.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Raw feature order expected by the scaler and SVGP model.
FEATURE_ORDER = ["x_m", "y_m", "decimal_year"]
RTK_FEATURE_ORDER = ["x_m", "y_m", "dec_year"]


In [ ]:
# ===========================================================
# SECTION 3 – LOAD & PARSE NOAA DATA
# ===========================================================
print(f"Loading {BEACH_LABEL} NOAA CSV ...")
df = pd.read_csv(NOAA_CSV)

required = {'x_m', 'y_m', 'elevation_m', 'decimal_year', 'acquisition_date_mid'}
missing  = required - set(df.columns)
if missing:
    raise ValueError(f"NOAA CSV is missing columns: {missing}")

df = df.dropna(subset=list(required))
print(f"  {len(df):,} valid NOAA points")
print(f"  Survey dates found:")
for date_str, dec_yr in sorted(
        df.groupby('acquisition_date_mid')['decimal_year'].first().items()):
    print(f"    {date_str}  →  {dec_yr:.6f}")

# Total NOAA dataset size. This is the correct SVGP num_data for the
# initial NOAA mini-batch training stage; RTK updates reset num_data per survey.
N_NOAA = len(df)
print(f"\n  N_NOAA = {N_NOAA:,}  (used for NOAA SVGP num_data)")

In [ ]:
# ===========================================================
# SECTION 4 – RTK FILE LOADER & DATE DISCOVERY
# ===========================================================

def _to_decimal_year(dt):
    if pd.isnull(dt):
        return float('nan')
    year_start = _datetime(dt.year, 1, 1)
    year_end   = _datetime(dt.year + 1, 1, 1)
    year_len   = (year_end - year_start).total_seconds()
    elapsed    = (dt.to_pydatetime() - year_start).total_seconds()
    return dt.year + elapsed / year_len


def load_rtk_file(path: str) -> pd.DataFrame:
    df_rtk = pd.read_csv(path)
    if 'fix_status' in df_rtk.columns:
        df_rtk = df_rtk[
            df_rtk['fix_status'].str.strip().str.upper() == 'FIXED'
        ].copy()
    df_rtk = df_rtk.rename(columns={
        'easting_m' : 'x_m',
        'northing_m': 'y_m',
    })
    df_rtk['dt_parsed'] = pd.to_datetime(df_rtk['datetime_local'], errors='coerce')
    df_rtk['dec_year']  = df_rtk['dt_parsed'].apply(_to_decimal_year)
    keep   = ['x_m', 'y_m', 'elevation_m', 'dec_year', 'dt_parsed']
    df_rtk = df_rtk.dropna(subset=keep).reset_index(drop=True)
    return df_rtk[keep]


def _get_survey_date(path: str) -> pd.Timestamp:
    try:
        tmp = pd.read_csv(path, usecols=['datetime_local']).dropna()
        if tmp.empty:
            return pd.Timestamp.max
        return pd.Timestamp(str(tmp['datetime_local'].iloc[0])[:10])
    except Exception:
        return pd.Timestamp.max


# ── Discover RTK files and extract date range NOW
# so the scaler in Section 5 covers the full temporal domain
_all_paths = sorted(
    glob.glob(os.path.join(RTK_DIR, '*.csv')),
    key=_get_survey_date
)

rtk_paths_by_date = {}
for p in _all_paths:
    survey_date = _get_survey_date(p)
    if survey_date == pd.Timestamp.max:
        continue
    date_key = survey_date.strftime('%Y-%m-%d')
    rtk_paths_by_date.setdefault(date_key, []).append(p)

duplicate_rtk_dates = {
    date: paths
    for date, paths in rtk_paths_by_date.items()
    if len(paths) > 1
}

if duplicate_rtk_dates:
    lines = [
        'Duplicate RTK survey date(s) found.',
        'Keep one converted RTK CSV per survey date before running streaming updates.',
    ]
    for date, paths in sorted(duplicate_rtk_dates.items()):
        lines.append(f'  {date}:')
        for p in sorted(paths):
            lines.append(f'    - {os.path.basename(p)}')
    raise ValueError('\n'.join(lines))

print(f"Found {len(_all_paths)} RTK file(s):")
for p in _all_paths:
    print(f"  {_get_survey_date(p).strftime('%Y-%m-%d')}  ←  {os.path.basename(p)}")

# Extract the maximum decimal year across all RTK files
rtk_dec_years = []
for p in _all_paths:
    tmp = pd.read_csv(p, usecols=['datetime_local']).dropna()
    if not tmp.empty:
        dt = pd.Timestamp(str(tmp['datetime_local'].iloc[0])[:10])
        rtk_dec_years.append(_to_decimal_year(dt))

RTK_DATE_MAX = max(rtk_dec_years) if rtk_dec_years else None
print(f"\n  RTK temporal range max : {RTK_DATE_MAX:.6f}")
print("RTK loader helpers and date discovery complete.")

In [ ]:
# ===========================================================
# SECTION 5 – BUILD TRAINING ARRAYS & SCALER
# ===========================================================

from sklearn.preprocessing import MinMaxScaler

X_raw = np.column_stack([
    df['x_m'].values,
    df['y_m'].values,
    df['decimal_year'].values
]).astype(np.float64)

Y = df['elevation_m'].values[:, None].astype(np.float64)

print(f"  X_raw : {X_raw.shape}  ({' | '.join(FEATURE_ORDER)})")
print(f"  Y     : {Y.shape}  (elevation_m)")
print(f"\n  Raw ranges:")
print(f"    x_m          [{X_raw[:,0].min():.1f}, {X_raw[:,0].max():.1f}]")
print(f"    y_m          [{X_raw[:,1].min():.1f}, {X_raw[:,1].max():.1f}]")
print(f"    decimal_year [{X_raw[:,2].min():.4f}, {X_raw[:,2].max():.4f}]")

# ── Define explicit domain bounds ─────────────────────────
# Spatial bounds come from the NOAA data extent.
# Temporal bounds cover the full range from first NOAA survey
# to the last RTK survey, ensuring no extrapolation.
x_min, x_max = X_raw[:,0].min(), X_raw[:,0].max()
y_min, y_max = X_raw[:,1].min(), X_raw[:,1].max()
t_min        = X_raw[:,2].min()
t_max        = RTK_DATE_MAX if RTK_DATE_MAX is not None else X_raw[:,2].max()

print(f"\n  Explicit domain bounds:")
print(f"    x_m          [{x_min:.1f}, {x_max:.1f}]")
print(f"    y_m          [{y_min:.1f}, {y_max:.1f}]")
print(f"    decimal_year [{t_min:.6f}, {t_max:.6f}]  (extended to RTK max)")

# Build a two-row array defining the explicit min/max bounds
# MinMaxScaler fit on these two rows maps exactly to [-1, +1]
domain_bounds = np.array([
    [x_min, y_min, t_min],
    [x_max, y_max, t_max]
])

scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(domain_bounds)
X_scaled = scaler.transform(X_raw)

print(f"\n  Scaled ranges (should be within [-1, +1] for NOAA):")
print(f"    x_s [{X_scaled[:,0].min():.3f}, {X_scaled[:,0].max():.3f}]")
print(f"    y_s [{X_scaled[:,1].min():.3f}, {X_scaled[:,1].max():.3f}]")
print(f"    t_s [{X_scaled[:,2].min():.3f}, {X_scaled[:,2].max():.3f}]")
print(f"\n  Scaler fit on explicit domain bounds.")
print(f"  RTK dates will scale to within [-1, +1] by construction.")


In [ ]:
# ===========================================================
# SECTION 6 – BUILD RASTER GRIDS (common spatial extent)
# ===========================================================
# Bin all survey epochs onto the same spatial grid
# so that elevation maps are spatially comparable across time.

x_global_min, x_global_max = X_raw[:,0].min(), X_raw[:,0].max()
y_global_min, y_global_max = X_raw[:,1].min(), X_raw[:,1].max()

x_edges   = np.linspace(x_global_min, x_global_max, TARGET_W + 1)
y_edges   = np.linspace(y_global_min, y_global_max, TARGET_H + 1)
x_centres = 0.5 * (x_edges[:-1] + x_edges[1:])
y_centres = 0.5 * (y_edges[:-1] + y_edges[1:])
xx, yy    = np.meshgrid(x_centres, y_centres)
COMMON_COORDS_RAW = np.column_stack([xx.ravel(), yy.ravel()])  # shape (H*W, 2)

maps, dec_years, date_labels = [], [], []

for date_str, grp in df.groupby('acquisition_date_mid'):
    dec_yr = grp['decimal_year'].iloc[0]
    x_vals = grp['x_m'].values
    y_vals = grp['y_m'].values
    z_vals = grp['elevation_m'].values

    xi = np.clip(np.digitize(x_vals, x_edges) - 1, 0, TARGET_W - 1)
    yi = np.clip(np.digitize(y_vals, y_edges) - 1, 0, TARGET_H - 1)

    z_sum   = np.zeros((TARGET_H, TARGET_W), dtype=np.float64)
    z_count = np.zeros((TARGET_H, TARGET_W), dtype=np.int32)
    np.add.at(z_sum,   (yi, xi), z_vals)
    np.add.at(z_count, (yi, xi), 1)

    with np.errstate(invalid='ignore'):
        z_grid = np.where(z_count > 0,
                          z_sum / z_count,
                          np.nan).astype(np.float32)

    maps.append(z_grid)
    dec_years.append(dec_yr)
    date_labels.append(date_str)

maps      = np.array(maps)
dec_years = np.array(dec_years)

print(f"  Grids built: {maps.shape}  (all on common {TARGET_H}×{TARGET_W} grid)")
print(f"  Spatial extent: x=[{x_global_min:.1f}, {x_global_max:.1f}]  "
      f"y=[{y_global_min:.1f}, {y_global_max:.1f}]")

In [ ]:
# ===========================================================
# SECTION 7 – TENSORFLOW DATA PIPELINE
# ===========================================================
dataset = (
    tf.data.Dataset.from_tensor_slices((X_scaled, Y))
    .shuffle(buffer_size=50_000)
    .batch(BATCH_SIZE)
    .repeat()
)
train_iter = iter(dataset)
print("TF dataset pipeline ready.")

In [ ]:
# ===========================================================
# SECTION 8 – SVGP MODEL (spatio-temporal product kernel)
# ===========================================================

ind_idx = np.random.choice(len(X_scaled), M_INDUCING, replace=False)
Z_ind   = X_scaled[ind_idx].copy()

# Separate spatial and temporal kernels
k_space = gpflow.kernels.Matern32(active_dims=[0, 1])   # (x_s, y_s)
k_time  = gpflow.kernels.Matern32(active_dims=[2])       # (t_s)
kernel  = k_space * k_time                               # product kernel

model = gpflow.models.SVGP(
    kernel=kernel,
    likelihood=gpflow.likelihoods.Gaussian(),
    inducing_variable=Z_ind,
    num_latent_gps=1,
    q_diag=False,
    num_data=N_NOAA,          # correct for the NOAA mini-batch training stage
)

print("\nModel summary:")
gpflow.utilities.print_summary(model)
print(f"\n  num_data = {model.num_data}  (NOAA training stage)")

In [ ]:
# ===========================================================
# SECTION 9 – INITIAL TRAINING ON NOAA DATA
# ===========================================================


optimizer = tf.optimizers.Adam(learning_rate=0.01)
elbo_log  = []

# Define the NOAA training step once outside the loop
@tf.function
def noaa_training_step():
    Xb, Yb = next(train_iter)
    with tf.GradientTape() as tape:
        loss = -model.elbo((Xb, Yb))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

print("Initial training on NOAA data ...")
for step in range(TRAIN_STEPS):
    loss     = noaa_training_step()
    elbo_val = -loss.numpy()
    elbo_log.append((step, elbo_val))
    if step % 200 == 0:
        print(f"  Step {step:4d}   ELBO: {elbo_val:.2f}")

print("NOAA training complete.")

elbo_arr = np.array(elbo_log)
plt.figure(figsize=(8, 4))
plt.plot(elbo_arr[:, 0], elbo_arr[:, 1], color='goldenrod')
plt.xlabel("Training step"); plt.ylabel("ELBO")
plt.title(f"{BEACH_LABEL} - NOAA Initial Training ELBO")
plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'elbo_curve_noaa.png'), dpi=150)
plt.show()

In [ ]:
# ===========================================================
# SECTION 10 – SAVE / LOAD HELPERS
# ===========================================================

INITIAL_MODEL_PATH = OUT_DIR / "svgp_initial_model.npz"
FINAL_MODEL_PATH = OUT_DIR / "svgp_final_model.npz"
METADATA_PATH = OUT_DIR / "model_metadata.json"
SURVEY_MANIFEST_PATH = OUT_DIR / "survey_manifest.csv"

per_survey_checkpoints = []


def _json_ready(value):
    """Convert NumPy, TensorFlow, and Path objects into JSON-safe values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, dict):
        return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(v) for v in value]
    return value


def _safe_checkpoint_stem(file_name):
    stem = Path(file_name).stem
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("_")
    return stem or "survey"


def survey_checkpoint_path(date_str, file_name):
    stem = _safe_checkpoint_stem(file_name)
    return OUT_DIR / f"svgp_after_{date_str}_{stem}.npz"


def save_svgp_model(mdl, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    params = gpflow.utilities.parameter_dict(mdl)
    np.savez(path, **{k: v.numpy() for k, v in params.items()})
    print(f"  Model saved → {path}")
    return path


def load_svgp_model(mdl, path):
    data   = np.load(path, allow_pickle=False)
    loaded = {k: tf.convert_to_tensor(v) for k, v in data.items()}
    gpflow.utilities.multiple_assign(mdl, loaded)
    print(f"  Model loaded ← {path}")
    return mdl


def _scaler_state():
    return {
        "class": scaler.__class__.__name__,
        "feature_range": list(scaler.feature_range),
        "feature_order": FEATURE_ORDER,
        "domain_bounds": {
            "x_m": {"min": x_min, "max": x_max},
            "y_m": {"min": y_min, "max": y_max},
            "decimal_year": {"min": t_min, "max": t_max},
        },
        "data_min_": scaler.data_min_,
        "data_max_": scaler.data_max_,
        "data_range_": scaler.data_range_,
        "scale_": scaler.scale_,
        "min_": scaler.min_,
    }


def save_survey_manifest(records, path=SURVEY_MANIFEST_PATH):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    manifest_df = pd.DataFrame(records)
    manifest_df.to_csv(path, index=False)
    print(f"  Survey manifest saved → {path}")
    return path


def save_run_metadata(
    model_stage,
    latest_assimilated_date=None,
    final_model_path=None,
    manifest_path=None,
    path=METADATA_PATH,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        "created_utc": _datetime.now(_timezone.utc).isoformat(timespec="seconds"),
        "model_stage": model_stage,
        "beach_key": BEACH,
        "beach_label": BEACH_LABEL,
        "project_root": PROJECT_ROOT,
        "paths": {
            "noaa_csv": NOAA_CSV,
            "rtk_dir": RTK_DIR,
            "out_dir": OUT_DIR,
            "initial_model": INITIAL_MODEL_PATH,
            "final_model": final_model_path,
            "survey_manifest": manifest_path,
        },
        "data": {
            "last_noaa_date": LAST_NOAA_DATE_STR,
            "n_noaa_points": N_NOAA,
            "rtk_files_discovered": [os.path.basename(p) for p in _all_paths],
            "latest_assimilated_date": latest_assimilated_date,
        },
        "features": {
            "model_feature_order": FEATURE_ORDER,
            "rtk_feature_order": RTK_FEATURE_ORDER,
            "target_column": "elevation_m",
        },
        "scaler": _scaler_state(),
        "hyperparameters": {
            "target_h": TARGET_H,
            "target_w": TARGET_W,
            "m_inducing": M_INDUCING,
            "batch_size": BATCH_SIZE,
            "train_steps": TRAIN_STEPS,
            "rtk_update_steps": RTK_UPDATE_STEPS,
            "random_seed": RANDOM_SEED,
        },
        "software": {
            "python": sys.version.split()[0],
            "tensorflow": tf.__version__,
            "gpflow": gpflow.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
        "checkpoints": {
            "initial_model": INITIAL_MODEL_PATH,
            "per_survey": per_survey_checkpoints,
            "final_model": final_model_path,
        },
    }
    path.write_text(json.dumps(_json_ready(metadata), indent=2), encoding="utf-8")
    print(f"  Run metadata saved → {path}")
    return path


print("Saving initial trained model ...")
save_svgp_model(model, INITIAL_MODEL_PATH)


In [ ]:
# ===========================================================
# SECTION 11 – PREDICTION HELPER
# ===========================================================


def predict_scaled(mdl, X_raw_input):
    """
    Scale raw (x_m, y_m, decimal_year) inputs and return
    (mean, std) in original elevation units (metres).
    Uses predict_y so reported sigma includes likelihood noise,
    making it directly comparable to RTK measurement error.
    """
    X_s   = scaler.transform(X_raw_input.astype(np.float64))
    mu, var = mdl.predict_y(tf.convert_to_tensor(X_s, dtype=tf.float64))
    return mu.numpy().ravel(), np.sqrt(var.numpy().ravel())

print("predict_scaled() helper defined.")

In [ ]:
# ===========================================================
# SECTION 12 – VALIDATION HELPER
# ===========================================================


def validate_model(mdl, rdf):
    """
    Compute MAE, RMSE, Bias, R², calibration stats for one RTK survey.
    Returns a dict including raw arrays for downstream plotting.
    sigma comes from predict_y (includes likelihood noise).
    """
    X_rtk_raw = np.column_stack([
        rdf['x_m'].values,
        rdf['y_m'].values,
        rdf['dec_year'].values
    ]).astype(np.float64)

    z_pred, sigma = predict_scaled(mdl, X_rtk_raw)
    z_obs         = rdf['elevation_m'].values
    res           = z_pred - z_obs

    mae  = float(np.mean(np.abs(res)))
    rmse = float(np.sqrt(np.mean(res**2)))
    bias = float(np.mean(res))
    ss_res = np.sum(res**2)
    ss_tot = np.sum((z_obs - np.mean(z_obs))**2)
    r2   = float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

    res_norm = res / (sigma + 1e-8)
    pct_1s   = float(np.mean(np.abs(res_norm) <= 1) * 100)
    pct_2s   = float(np.mean(np.abs(res_norm) <= 2) * 100)

    return {
        'z_obs'   : z_obs,
        'z_pred'  : z_pred,
        'sigma'   : sigma,
        'res'     : res,
        'res_norm': res_norm,
        'MAE_m'   : round(mae,   4),
        'RMSE_m'  : round(rmse,  4),
        'Bias_m'  : round(bias,  4),
        'R2'      : round(r2,    4),
        'pct_1s'  : round(pct_1s, 1),
        'pct_2s'  : round(pct_2s, 1),
        'n'       : len(rdf),
    }

print("validate_model() helper defined.")

In [ ]:
# ===========================================================
# SECTION 13 – RTK VALIDATION + ONLINE MODEL UPDATE
# ===========================================================

# _all_paths already discovered and sorted in Section 4
print(f"Processing {len(_all_paths)} RTK file(s) — chronological order:")
for p in _all_paths:
    print(f"  {_get_survey_date(p).strftime('%Y-%m-%d')}  ←  {os.path.basename(p)}")

# ── RTK update optimiser (separate from NOAA optimiser) ────
rtk_optimizer = tf.optimizers.Adam(learning_rate=0.001)

# — single function, no loop-redefine, args passed explicitly
def make_rtk_update_fn(mdl, opt, rtk_iter_ref):

    @tf.function(reduce_retracing=True)
    def _step(Xb, Yb):
        with tf.GradientTape() as tape:
            loss = -mdl.elbo((Xb, Yb))
        grads = tape.gradient(loss, mdl.trainable_variables)
        opt.apply_gradients(zip(grads, mdl.trainable_variables))
        return loss
    return _step

# Containers for held-out validation, streaming update diagnostics, and manifests.
rtk_bias_log             = []
all_validation_results   = []
streaming_update_records = []
all_z_obs_pooled         = []
all_z_pred_pooled        = []
survey_manifest_records = [{
    'beach': BEACH,
    'source_type': 'NOAA_combined',
    'file': os.path.basename(NOAA_CSV),
    'path': str(NOAA_CSV),
    'survey_date': 'multiple',
    'decimal_year': '',
    'n_points_used': N_NOAA,
    'status': 'initial_training_data',
    'checkpoint_after_update': str(INITIAL_MODEL_PATH),
}]

# Containers for section-16 held-out calibration:
# We record residuals from Step A (before update) — this is the
# only honest held-out measurement.
held_out_res_norm  = []
held_out_sigma     = []

last_noaa_dt = _datetime.strptime(LAST_NOAA_DATE_STR, '%Y-%m-%d')
prev_date    = last_noaa_dt
latest_assimilated_date = None


def _grid_sigma_summary(mdl, dec_year):
    """Predict uncertainty on the common spatial grid at one decimal-year timestamp."""
    X_grid_raw = np.column_stack([
        COMMON_COORDS_RAW,
        np.full(COMMON_COORDS_RAW.shape[0], float(dec_year))
    ]).astype(np.float64)
    _, sigma_grid = predict_scaled(mdl, X_grid_raw)
    return {
        'mean': float(np.mean(sigma_grid)),
        'median': float(np.median(sigma_grid)),
        'p90': float(np.percentile(sigma_grid, 90)),
    }

for rpath in _all_paths:
    fname = os.path.basename(rpath)
    print(f"\n{'='*60}")
    print(f"Processing: {fname}")
    print(f"{'='*60}")

    rdf = load_rtk_file(rpath)
    if len(rdf) < 5:
        skipped_date = _get_survey_date(rpath).strftime('%Y-%m-%d')
        print(f"  SKIP — not enough FIXED points ({len(rdf)})")
        survey_manifest_records.append({
            'beach': BEACH,
            'source_type': 'RTK',
            'file': fname,
            'path': str(rpath),
            'survey_date': skipped_date,
            'decimal_year': '',
            'n_points_used': len(rdf),
            'status': 'skipped_too_few_fixed_points',
            'checkpoint_after_update': '',
        })
        continue

    date_str    = rdf['dt_parsed'].iloc[0].strftime('%Y-%m-%d')
    survey_date = rdf['dt_parsed'].iloc[0].to_pydatetime().replace(tzinfo=None)
    gap_days    = (survey_date - prev_date).days

    print(f"  Date     : {date_str}")
    print(f"  Gap      : {gap_days} days since previous survey")
    print(f"  N points : {len(rdf)}")

    survey_dec_year = float(rdf['dec_year'].median())
    grid_sigma_before = _grid_sigma_summary(model, survey_dec_year)

    # ── STEP A — VALIDATE (model has NOT seen this RTK data yet) ──
    print("\n  [A] Validating (held-out) ...")
    m = validate_model(model, rdf)

    print(f"      MAE  = {m['MAE_m']:.4f} m")
    print(f"      RMSE = {m['RMSE_m']:.4f} m")
    print(f"      Bias = {m['Bias_m']:+.4f} m")
    print(f"      R²   = {m['R2']:.4f}")
    print(f"      Held-out mean σ = {np.mean(m['sigma']):.4f} m")
    print(f"      Grid mean σ before update = {grid_sigma_before['mean']:.4f} m")

    # Collect calibration residuals from the held-out pre-update prediction.
    held_out_res_norm.extend(m['res_norm'])
    held_out_sigma.extend(m['sigma'])

    rtk_bias_log.append({
        'date'    : date_str,
        'bias_m'  : m['Bias_m'],
        'gap_days': gap_days,
    })
    prev_date = survey_date

    all_z_obs_pooled.extend(m['z_obs'])
    all_z_pred_pooled.extend(m['z_pred'])
    all_validation_results.append({
        'file'    : fname,
        'date'    : date_str,
        'gap_days': gap_days,
        'n_points': m['n'],
        'MAE_m'   : m['MAE_m'],
        'RMSE_m'  : m['RMSE_m'],
        'Bias_m'  : m['Bias_m'],
        'R2'      : m['R2'],
        'heldout_mean_sigma_m'  : round(float(np.mean(m['sigma'])), 4),
        'heldout_median_sigma_m': round(float(np.median(m['sigma'])), 4),
        'heldout_p90_sigma_m'   : round(float(np.percentile(m['sigma'], 90)), 4),
        'heldout_pct_1s'        : m['pct_1s'],
        'heldout_pct_2s'        : m['pct_2s'],
        'metric_stage'          : 'heldout_before_update',
    })

    # Scatter + spatial residual plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{BEACH_LABEL} RTK Validation (held-out) - {date_str}", fontsize=12)
    lims = [min(m['z_obs'].min(), m['z_pred'].min()) - 0.1,
            max(m['z_obs'].max(), m['z_pred'].max()) + 0.1]
    axes[0].scatter(m['z_obs'], m['z_pred'], s=15, alpha=0.6, color='steelblue')
    axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='1:1')
    axes[0].set_xlabel('RTK Observed (m)'); axes[0].set_ylabel('SVGP Predicted (m)')
    axes[0].set_title(f'Predicted vs Observed\n'
                      f'MAE={m["MAE_m"]:.3f}m  RMSE={m["RMSE_m"]:.3f}m  R²={m["R2"]:.3f}')
    axes[0].set_xlim(lims); axes[0].set_ylim(lims)
    axes[0].legend(); axes[0].grid(True)

    rlim = np.percentile(np.abs(m['res']), 98)
    sc   = axes[1].scatter(rdf['x_m'].values, rdf['y_m'].values,
                           c=m['res'], cmap='RdBu', s=20,
                           vmin=-rlim, vmax=rlim)
    plt.colorbar(sc, ax=axes[1], label='Residual (m)')
    axes[1].set_title('Spatial Residuals (Predicted − Observed)')
    axes[1].set_xlabel('Easting (m)'); axes[1].set_ylabel('Northing (m)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'rtk_validation_{date_str}.png'), dpi=150)
    plt.show()

    # ── STEP B — ONLINE UPDATE ────────────────────────────────────
    print(f"\n  [B] Updating model ({RTK_UPDATE_STEPS} steps) ...")
    X_rtk_raw = np.column_stack([
        rdf['x_m'].values,
        rdf['y_m'].values,
        rdf['dec_year'].values
    ]).astype(np.float64)
    X_rtk_s = scaler.transform(X_rtk_raw)
    Y_rtk   = rdf['elevation_m'].values[:, None].astype(np.float64)

    rtk_ds = (
        tf.data.Dataset.from_tensor_slices((X_rtk_s, Y_rtk))
        .shuffle(buffer_size=max(len(rdf), 1), seed=RANDOM_SEED, reshuffle_each_iteration=True)
        .batch(min(BATCH_SIZE, len(rdf)))
        .repeat()
    )
    rtk_iter_local = iter(rtk_ds)

    # Build the update step once per file, passing model/optimizer explicitly.
    _rtk_step = make_rtk_update_fn(model, rtk_optimizer, rtk_iter_local)

    # During RTK-only updates, each mini-batch represents the current RTK
    # survey, not the full NOAA training set used for initial training.
    noaa_num_data = model.num_data
    model.num_data = len(rdf)
    print(f"      RTK ELBO num_data = {model.num_data} (restoring NOAA num_data after update)")

    update_elbo_log = []
    try:
        for step in range(RTK_UPDATE_STEPS):
            Xb, Yb   = next(rtk_iter_local)
            loss     = _rtk_step(Xb, Yb)
            elbo_val = -loss.numpy()
            update_elbo_log.append(elbo_val)
            if step % 200 == 0:
                print(f"      Update step {step:4d}   ELBO: {elbo_val:.2f}")
    finally:
        model.num_data = noaa_num_data

    print(f"  Model updated with {date_str} RTK data.")

    checkpoint_path = survey_checkpoint_path(date_str, fname)
    save_svgp_model(model, checkpoint_path)
    latest_assimilated_date = date_str
    per_survey_checkpoints.append({
        'date': date_str,
        'file': fname,
        'path': str(checkpoint_path),
        'n_points': m['n'],
    })

    grid_sigma_after = _grid_sigma_summary(model, survey_dec_year)
    streaming_update_records.append({
        'file'    : fname,
        'date'    : date_str,
        'gap_days': gap_days,
        'n_points': m['n'],
        'heldout_MAE_m'   : m['MAE_m'],
        'heldout_RMSE_m'  : m['RMSE_m'],
        'heldout_Bias_m'  : m['Bias_m'],
        'heldout_R2'      : m['R2'],
        'heldout_mean_sigma_m'  : round(float(np.mean(m['sigma'])), 4),
        'heldout_median_sigma_m': round(float(np.median(m['sigma'])), 4),
        'heldout_p90_sigma_m'   : round(float(np.percentile(m['sigma'], 90)), 4),
        'heldout_pct_1s'        : m['pct_1s'],
        'heldout_pct_2s'        : m['pct_2s'],
        'grid_mean_sigma_before_m'  : round(grid_sigma_before['mean'], 4),
        'grid_median_sigma_before_m': round(grid_sigma_before['median'], 4),
        'grid_p90_sigma_before_m'   : round(grid_sigma_before['p90'], 4),
        'grid_mean_sigma_after_m'   : round(grid_sigma_after['mean'], 4),
        'grid_median_sigma_after_m' : round(grid_sigma_after['median'], 4),
        'grid_p90_sigma_after_m'    : round(grid_sigma_after['p90'], 4),
        'grid_sigma_reduction_m'    : round(grid_sigma_before['mean'] - grid_sigma_after['mean'], 4),
        'grid_sigma_reduction_pct'  : round(
            ((grid_sigma_before['mean'] - grid_sigma_after['mean']) / grid_sigma_before['mean']) * 100, 2
        ) if grid_sigma_before['mean'] > 0 else np.nan,
        'checkpoint_after_update'   : str(checkpoint_path),
        'update_stage'              : 'after_update_checkpointed',
    })
    survey_manifest_records.append({
        'beach': BEACH,
        'source_type': 'RTK',
        'file': fname,
        'path': str(rpath),
        'survey_date': date_str,
        'decimal_year': round(survey_dec_year, 6),
        'n_points_used': m['n'],
        'status': 'validated_then_assimilated',
        'checkpoint_after_update': str(checkpoint_path),
    })

    print(f"      Grid mean σ after update  = {grid_sigma_after['mean']:.4f} m")
    print(f"      Grid mean σ reduction     = {grid_sigma_before['mean'] - grid_sigma_after['mean']:+.4f} m")

    plt.figure(figsize=(7, 3))
    plt.plot(update_elbo_log, color='steelblue')
    plt.xlabel("Update step"); plt.ylabel("ELBO")
    plt.title(f"Online Update ELBO – {date_str}")
    plt.grid(True); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'elbo_update_{date_str}.png'), dpi=150)
    plt.show()

print("\nSaving final streaming model state and run records ...")
save_svgp_model(model, FINAL_MODEL_PATH)
save_survey_manifest(survey_manifest_records, SURVEY_MANIFEST_PATH)
save_run_metadata(
    model_stage='final_streaming_model',
    latest_assimilated_date=latest_assimilated_date,
    final_model_path=FINAL_MODEL_PATH,
    manifest_path=SURVEY_MANIFEST_PATH,
    path=METADATA_PATH,
)


In [ ]:
# ===========================================================
# SECTION 14 – OVERALL VALIDATION SUMMARY
# ===========================================================


results_df = pd.DataFrame(all_validation_results)
print("\n=== RTK Validation Summary (held-out, before each update) ===")
print(results_df[['date', 'gap_days', 'n_points',
                   'MAE_m', 'RMSE_m', 'Bias_m', 'R2']].to_string(index=False))
results_df.to_csv(os.path.join(OUT_DIR, 'rtk_validation_summary.csv'), index=False)

all_z_obs_pooled  = np.array(all_z_obs_pooled)
all_z_pred_pooled = np.array(all_z_pred_pooled)
all_res_pooled    = all_z_pred_pooled - all_z_obs_pooled

ss_res     = np.sum(all_res_pooled**2)
ss_tot     = np.sum((all_z_obs_pooled - np.mean(all_z_obs_pooled))**2)
r2_overall = 1 - ss_res / ss_tot

print(f"\n  Overall MAE  = {np.mean(np.abs(all_res_pooled)):.4f} m")
print(f"  Overall RMSE = {np.sqrt(np.mean(all_res_pooled**2)):.4f} m")
print(f"  Overall Bias = {np.mean(all_res_pooled):+.4f} m")
print(f"  Overall R²   = {r2_overall:.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_z_obs_pooled, all_z_pred_pooled, s=10, alpha=0.4, color='steelblue')
lims = [min(all_z_obs_pooled.min(), all_z_pred_pooled.min()) - 0.1,
        max(all_z_obs_pooled.max(), all_z_pred_pooled.max()) + 0.1]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='1:1 line')
ax.set_xlabel('RTK Observed Elevation (m)')
ax.set_ylabel('SVGP Predicted Elevation (m)')
ax.set_title(f'{BEACH_LABEL} - All RTK Surveys Pooled (held-out)\n'
             f'MAE={np.mean(np.abs(all_res_pooled)):.3f}m  '
             f'RMSE={np.sqrt(np.mean(all_res_pooled**2)):.3f}m  '
             f'R²={r2_overall:.3f}')
ax.legend(); ax.grid(True)
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'rtk_scatter_pooled.png'), dpi=150)
plt.show()

print("\n=== Bias Log ===")
for entry in rtk_bias_log:
    rate = abs(entry['bias_m']) / max(entry['gap_days'], 1)
    print(f"  {entry['date']}  bias={entry['bias_m']:+.4f}m  "
          f"gap={entry['gap_days']}d  rate={rate:.6f} m/day  "
          f"({rate*14:.4f} m/2weeks)")

In [ ]:
# ===========================================================
# SECTION 15 - PREDICTION SURFACE  (latest assimilated epoch, common grid)
# ===========================================================

print("\nBuilding prediction surface for latest assimilated epoch ...")


def _latest_prediction_epoch():
    """Return (decimal_year, date_label, source_label) for the final surface."""
    if all_validation_results:
        latest_date = max(pd.Timestamp(r["date"]) for r in all_validation_results)
        matching_paths = [
            p for p in _all_paths
            if _get_survey_date(p).normalize() == latest_date.normalize()
        ]
        if matching_paths:
            latest_rdf = load_rtk_file(matching_paths[-1])
            if not latest_rdf.empty:
                latest_dt = latest_rdf["dt_parsed"].max()
                return (
                    float(latest_rdf["dec_year"].max()),
                    latest_dt.strftime("%Y-%m-%d"),
                    "latest RTK update",
                )

    noaa_idx = int(np.nanargmax(dec_years))
    return float(dec_years[noaa_idx]), str(date_labels[noaa_idx]), "latest NOAA survey"


t_latest, latest_date_label, latest_source = _latest_prediction_epoch()
print(f"  Prediction date : {latest_date_label} ({latest_source})")
print(f"  Decimal year    : {t_latest:.6f}")

t_col      = np.full((COMMON_COORDS_RAW.shape[0], 1), t_latest)
X_pred_raw = np.column_stack([COMMON_COORDS_RAW, t_col]).astype(np.float64)

mu_pred, sig_pred = predict_scaled(model, X_pred_raw)

H, W       = TARGET_H, TARGET_W
mean_map   = mu_pred.reshape(H, W)
uncert_map = sig_pred.reshape(H, W)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{BEACH_LABEL} SVGP Prediction - {latest_date_label}", fontsize=12)

extent = [x_global_min, x_global_max, y_global_min, y_global_max]

im0 = axes[0].imshow(mean_map, cmap='terrain', extent=extent,
                     origin='lower', aspect='auto')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)')
axes[0].set_title('Mean Elevation')
axes[0].set_xlabel('Easting (m)'); axes[0].set_ylabel('Northing (m)')

im1 = axes[1].imshow(uncert_map, cmap='inferno', extent=extent,
                     origin='lower', aspect='auto')
plt.colorbar(im1, ax=axes[1], label='1σ Uncertainty (m)')
axes[1].set_title('Predictive Uncertainty (1σ)')
axes[1].set_xlabel('Easting (m)'); axes[1].set_ylabel('Northing (m)')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'prediction_surface_latest.png'), dpi=150)
plt.show()


In [ ]:
# ===========================================================
# SECTION 16 – UNCERTAINTY CALIBRATION DIAGNOSTICS 
# ===========================================================

print("\n=== Uncertainty Calibration  [held-out predict_y] ===")

all_r_norm    = np.array(held_out_res_norm)
all_sigma_cal = np.array(held_out_sigma)

pct_1s    = np.mean(np.abs(all_r_norm) <= 1) * 100
pct_2s    = np.mean(np.abs(all_r_norm) <= 2) * 100
sharpness = np.mean(all_sigma_cal)

print(f"  Percent within ±1σ  = {pct_1s:.1f}%  (ideal ≈ 68%)")
print(f"  Percent within ±2σ  = {pct_2s:.1f}%  (ideal ≈ 95%)")
print(f"  Mean σ (sharpness)  = {sharpness:.4f} m")
print(f"  Std(norm residual)  = {np.std(all_r_norm):.3f}  (ideal ≈ 1)")
print(f"\n  NOTE: stats measured BEFORE each RTK update — genuine held-out.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"{BEACH_LABEL} - Uncertainty Calibration  [held-out, predict_y]", fontsize=12)

axes[0].hist(all_r_norm, bins=60, density=True, alpha=0.75, color='steelblue')
axes[0].axvline( 0, color='k', linewidth=2)
axes[0].axvline( 1, color='r', linestyle='--', label='±1σ')
axes[0].axvline(-1, color='r', linestyle='--')
axes[0].axvline( 2, color='r', linestyle=':',  label='±2σ')
axes[0].axvline(-2, color='r', linestyle=':')
x_range = np.linspace(-5, 5, 200)
axes[0].plot(x_range, _norm.pdf(x_range), 'k--', linewidth=1.5, label='N(0,1) ideal')
axes[0].set_xlabel('Normalised Residual  (pred − obs) / σ')
axes[0].set_ylabel('Probability Density')
axes[0].set_title(f'Normalised Residuals (held-out)\n'
                  f'±1σ={pct_1s:.1f}%  ±2σ={pct_2s:.1f}%  '
                  f'std={np.std(all_r_norm):.2f}')
axes[0].set_xlim(-6, 6); axes[0].legend(); axes[0].grid(True)

pit_vals = _norm.cdf(all_r_norm)
axes[1].hist(pit_vals, bins=20, density=True, alpha=0.75, color='goldenrod')
axes[1].axhline(1.0, color='k', linestyle='--', label='Ideal uniform')
axes[1].set_xlabel('PIT value'); axes[1].set_ylabel('Density')
axes[1].set_title('PIT Histogram  (flat = well-calibrated)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'calibration_diagnostics.png'), dpi=150)
plt.show()

In [ ]:
# ===========================================================
# SECTION 17 - STREAMING UPDATE IMPACT AND FORECAST STALENESS
# ===========================================================

print("\n=== Streaming Update Impact and Forecast Staleness ===")

analysis_label = globals().get(
    'BEACH_LABEL',
    'Bicentennial' if globals().get('BEACH') == 'bicentennial' else 'Bass'
)

update_impact_df = pd.DataFrame(streaming_update_records).copy()
if update_impact_df.empty:
    raise ValueError("No streaming update records found. Run Section 13 before this analysis.")

required_cols = {
    'date', 'gap_days', 'n_points', 'heldout_MAE_m', 'heldout_RMSE_m',
    'heldout_Bias_m', 'heldout_R2', 'heldout_mean_sigma_m',
    'grid_mean_sigma_before_m', 'grid_mean_sigma_after_m',
    'grid_sigma_reduction_m', 'grid_sigma_reduction_pct',
    'checkpoint_after_update', 'update_stage'
}
missing_cols = required_cols - set(update_impact_df.columns)
if missing_cols:
    raise ValueError(
        "Streaming update fields are missing. Rerun Section 13 after this notebook update. "
        f"Missing columns: {sorted(missing_cols)}"
    )

update_impact_df['date_dt'] = pd.to_datetime(update_impact_df['date'])
update_impact_df['abs_heldout_bias_m'] = update_impact_df['heldout_Bias_m'].abs()
update_impact_df = update_impact_df.sort_values('date_dt').reset_index(drop=True)

impact_cols = [
    'date', 'gap_days', 'n_points', 'heldout_MAE_m', 'heldout_RMSE_m',
    'heldout_Bias_m', 'heldout_R2', 'heldout_mean_sigma_m',
    'grid_mean_sigma_before_m', 'grid_mean_sigma_after_m',
    'grid_sigma_reduction_m', 'grid_sigma_reduction_pct',
    'checkpoint_after_update', 'update_stage'
]

print("\n--- Streaming Update Impact ---")
print(update_impact_df[impact_cols].to_string(index=False))
impact_path = os.path.join(OUT_DIR, 'streaming_update_impact.csv')
update_impact_df.drop(columns=['date_dt']).to_csv(impact_path, index=False)
print(f"\nSaved update-impact table -> {impact_path}")

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
fig.suptitle(f'{analysis_label} - Streaming Update Impact', fontsize=13)

x_dates = update_impact_df['date_dt']
axes[0].plot(x_dates, update_impact_df['heldout_MAE_m'], marker='o', label='Held-out MAE')
axes[0].plot(x_dates, update_impact_df['heldout_RMSE_m'], marker='o', label='Held-out RMSE')
axes[0].set_ylabel('Held-out error (m)')
axes[0].set_title('Pre-update held-out prediction error')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].bar(x_dates, update_impact_df['heldout_Bias_m'], width=8, color='steelblue', alpha=0.75)
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_ylabel('Bias (m)')
axes[1].set_title('Pre-update bias: model drift relative to new RTK survey')
axes[1].grid(True, axis='y', alpha=0.4)
for x, gap in zip(x_dates, update_impact_df['gap_days']):
    axes[1].annotate(f'{int(gap)}d', (x, 0), textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8, color='dimgray')

axes[2].plot(x_dates, update_impact_df['grid_mean_sigma_before_m'], marker='o', label='before update')
axes[2].plot(x_dates, update_impact_df['grid_mean_sigma_after_m'], marker='o', label='after update')
axes[2].set_ylabel('Mean grid sigma (m)')
axes[2].set_title('Common-grid predictive uncertainty at each survey date')
axes[2].legend(); axes[2].grid(True, alpha=0.4)
axes[2].set_xlabel('RTK survey date')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
impact_fig_path = os.path.join(OUT_DIR, 'streaming_update_impact.png')
plt.savefig(impact_fig_path, dpi=150)
plt.show()
print(f"Saved update-impact plot -> {impact_fig_path}")

# Forecast staleness: keep the final streaming model fixed and ask how its
# predictive uncertainty behaves as we evaluate farther beyond the latest RTK date.
latest_update_date = update_impact_df['date_dt'].max()
forecast_horizons_days = [0, 14, 30, 60, 90]
staleness_rows = []

for days_after in forecast_horizons_days:
    eval_dt = latest_update_date + pd.Timedelta(days=days_after)
    t_eval = _to_decimal_year(eval_dt)
    X_eval_raw = np.column_stack([
        COMMON_COORDS_RAW,
        np.full(COMMON_COORDS_RAW.shape[0], t_eval)
    ]).astype(np.float64)
    _, sigma_eval = predict_scaled(model, X_eval_raw)
    staleness_rows.append({
        'model_stage'       : 'final_streaming_model',
        'reference_date'    : latest_update_date.strftime('%Y-%m-%d'),
        'eval_date'         : eval_dt.strftime('%Y-%m-%d'),
        'days_after_update' : days_after,
        'decimal_year'      : round(float(t_eval), 6),
        'mean_sigma_m'      : round(float(np.mean(sigma_eval)), 4),
        'median_sigma_m'    : round(float(np.median(sigma_eval)), 4),
        'p90_sigma_m'       : round(float(np.percentile(sigma_eval, 90)), 4),
        'min_sigma_m'       : round(float(np.min(sigma_eval)), 4),
        'max_sigma_m'       : round(float(np.max(sigma_eval)), 4),
    })

staleness_df = pd.DataFrame(staleness_rows)
print("\n--- Forecast Staleness From Final Streaming Model ---")
print(staleness_df.to_string(index=False))
staleness_path = os.path.join(OUT_DIR, 'forecast_staleness_summary.csv')
staleness_df.to_csv(staleness_path, index=False)
print(f"\nSaved forecast-staleness table -> {staleness_path}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(staleness_df['days_after_update'], staleness_df['mean_sigma_m'], marker='o', label='mean sigma')
ax.plot(staleness_df['days_after_update'], staleness_df['p90_sigma_m'], marker='o', label='p90 sigma')
ax.set_xlabel('Days after latest RTK update')
ax.set_ylabel('Predictive uncertainty sigma (m)')
ax.set_title(f'{analysis_label} - Forecast Staleness After Latest RTK Update')
ax.grid(True, alpha=0.4); ax.legend()
plt.tight_layout()
staleness_fig_path = os.path.join(OUT_DIR, 'forecast_staleness.png')
plt.savefig(staleness_fig_path, dpi=150)
plt.show()
print(f"Saved forecast-staleness plot -> {staleness_fig_path}")
